# 7교시 · 복습과 정리

### — 오늘 배운 것을 열 가지로 정리합니다

**이 시간에 하는 일**

1. 오늘 쓴 기능을 **한 줄씩 다시 손으로 쳐 봅니다**
2. 어떤 상황에 무엇을 쓰는지 **말로 붙여 봅니다**
3. 회사에 돌아가서 **무엇부터 해 볼지** 정합니다

> 새로 배우는 것은 없습니다. 빈칸을 채우면서 오늘 하루를 되짚습니다.

## 오늘 쓰는 데이터 — Superstore 주문 내역

미국의 문구·가구 유통사 **Superstore** 의 주문 내역입니다.

- **한 행 = 주문에 담긴 품목 하나** (주문 하나에 품목이 여럿이면 여러 행)
- 기간: 2023년 ~ 2026년, 약 1만 행
- 주요 열 — `Order Date`(주문일) · `Region`(지역) · `Category`(대분류) ·
  `Sub-Category`(세부 품목) · `Sales`(매출) · `Quantity`(수량) ·
  `Discount`(할인율) · `Profit`(이익)

반품 여부는 `superstore_returns.csv` 에 **따로** 있습니다 (`Order ID`, `Returned`).

---
# ① Pandas — 표를 다루는 라이브러리

엑셀의 시트에 해당하는 것이 **DataFrame** 이고,
그것을 다루는 도구가 **Pandas** 입니다.

`import pandas as pd` 는 **"Pandas 를 pd 라는 이름으로 쓰겠다"** 는 뜻입니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/JasonWhiteLee/ak-data-analysis-basics/main/'

print(pd.__version__)

---
# ② `read_csv()` — 파일 불러오기

| 파일 | 함수 |
|---|---|
| CSV | `pd.read_csv()` |
| 엑셀 | `pd.read_excel()` |

`parse_dates` 를 적어 주면 날짜 열을 **글자가 아니라 날짜로** 읽습니다.

In [ ]:
orders = pd.read_csv(BASE + 'superstore_orders.csv',
                       parse_dates=['Order Date', 'Ship Date'])

print(orders.shape)

---
# ③ Jupyter Notebook — 오늘 하루 쓴 화면

지금 이 화면입니다. **Colab** 은 Jupyter Notebook 을 구글 서버에서 돌려주는 서비스입니다.

- **셀 단위**로 실행합니다. 전체를 한 번에만 돌리는 것이 아닙니다
- **코드와 결과를 한 화면**에서 봅니다. 그래프도 바로 아래에 나옵니다
- 엑셀 파일도 불러옵니다 (`pd.read_excel()`)

아래 셀을 실행하면 표가 셀 바로 아래에 나옵니다. 그것이 이 도구의 전부입니다.

In [ ]:
orders.head(3)

---
# ④ 결측치 — 비어 있는 값

`isna()` 와 `isnull()` 은 **같은 기능**입니다. 이름만 둘입니다.
열별로 몇 개나 비었는지는 `.sum()` 을 붙여서 봅니다.

In [ ]:
orders.isnull().sum()

`Ship Date` 가 비어 있었습니다. **그냥 지우면 안 됩니다** — 2교시에서 봤듯이
빈 값이 반품 건에 몰려 있어서, 지우면 반품률이 조용히 낮아집니다.

---
# ⑤ 이상치 — 유난히 크거나 작은 값

결측치는 **비어 있는 값**, 이상치는 **유난히 튀는 값**입니다. 다릅니다.

이상치는 **항상 지우는 것이 아닙니다.** 입력 오류일 수도 있고,
진짜 일어난 큰 거래일 수도 있습니다. 보고 판단합니다.

In [ ]:
orders.nlargest(5, 'Sales')[['Sub-Category', 'Sales', 'Quantity', 'Profit']]

---
# ⑥ `groupby()` — 항목별로 묶어서 계산

엑셀의 **피벗테이블**에 해당합니다. **묶을 기준**과 **계산 방법**을 정합니다.

In [ ]:
orders.groupby('Region')['Sales'].sum().round(0)

### 패턴 — 묶는 기준만 바꿔 가며

| 하고 싶은 것 | 코드 |
|---|---|
| 지역별 매출 합계 | `groupby('Region')['Sales'].sum()` |
| 대분류별 이익 평균 | `groupby('Category')['Profit'].mean()` |
| 세부 품목별 주문 건수 | `groupby('Sub-Category').size()` |
| 지역 × 대분류 | `groupby(['Region', 'Category'])['Sales'].sum()` |

아래에서 **대분류별 이익 합계**를 구해 보세요.

In [ ]:
orders.groupby('Category')['Profit'].sum().round(0)

---
# ⑦ `merge()` — 두 표를 공통 열로 합치기

엑셀의 **VLOOKUP** 에 해당합니다. `on=` 에 **양쪽에 다 있는 열 이름**을 적습니다.

In [ ]:
returns = pd.read_csv(BASE + 'superstore_returns.csv')

merged = pd.merge(orders, returns, on='Order ID', how='left')

print(merged.shape)

`how='left'` 는 **왼쪽 표(orders)의 행을 모두 남긴다**는 뜻입니다.
반품 기록이 없는 주문은 `Returned` 가 비어 있게 됩니다.

---
# ⑧ 그래프 — 무엇을 말하려는지에 따라 정해집니다

| 말하려는 것 | 그래프 |
|---|---|
| **시간에 따른 추이** | **선 그래프** |
| 항목 사이의 크기 비교 | 막대그래프 |
| 두 숫자의 관계 | 산점도 |
| 값이 퍼진 정도 | 히스토그램 · 상자그림 |

월별 매출 추이는 **선 그래프**입니다.

In [ ]:
monthly = orders.set_index('Order Date')['Sales'].resample('ME').sum()

plt.figure(figsize=(10, 3))
monthly.plot(kind='line')
plt.title('Monthly Sales')
plt.ylim(0)
plt.show()

`plt.ylim(0)` 으로 **세로축을 0부터** 그렸습니다.
5교시에서 봤듯이, 축을 자르면 같은 데이터가 전혀 다른 인상을 줍니다.

---
# ⑨ 상관계수 — 같이 움직이는 정도

**-1 에서 +1 사이**의 값입니다.

| 값 | 뜻 |
|---|---|
| +1 에 가깝다 | 한쪽이 오르면 다른 쪽도 오른다 |
| 0 에 가깝다 | 직선 관계가 약하다 |
| -1 에 가깝다 | 한쪽이 오르면 다른 쪽은 내린다 |

In [ ]:
print(round(orders['Discount'].corr(orders['Profit']), 3))

---
# ⑩ 상관은 인과가 아닙니다

할인율과 이익의 상관계수는 **-0.219** 입니다. 음의 관계입니다.
그렇다고 **"할인을 없애면 이익이 오른다"** 고 말할 수는 없습니다.

- 원래 안 팔리는 품목이라 할인을 했을 수도 있습니다 (**순서가 반대**)
- 재고가 오래된 품목이라 할인도 하고 이익도 낮을 수도 있습니다 (**숨은 원인**)

## 오늘 하루의 세 가지 주의

| | |
|---|---|
| **상관 ≠ 인과** | 같이 움직인다고 원인은 아닙니다 |
| **평균만 보면 안 됩니다** | 평균 239 뒤에 중앙값 54 가 있었습니다 |
| **이상치를 항상 지우면 안 됩니다** | 오류일 수도, 진짜 큰 거래일 수도 있습니다 |

In [ ]:
print('평균  :', round(orders['Sales'].mean(), 1))
print('중앙값:', round(orders['Sales'].median(), 1))
print('평균보다 적은 주문 비율: {:.1f}%'.format(
    (orders['Sales'] < orders['Sales'].mean()).mean() * 100))

---
# 열 가지를 한 장에

| | | |
|---|---|---|
| ① | **Pandas** | 표를 다루는 라이브러리 |
| ② | **`read_csv()`** | 파일 불러오기 (엑셀은 `read_excel()`) |
| ③ | **Jupyter Notebook** | 코드와 결과를 한 화면에서, 셀 단위로 실행 |
| ④ | **결측치** | 비어 있는 값 — `isna()` · `isnull()` |
| ⑤ | **이상치** | 유난히 크거나 작은 값 |
| ⑥ | **`groupby()`** | 항목별로 묶어서 계산 (피벗테이블) |
| ⑦ | **`merge()`** | 두 표를 공통 열로 합치기 (VLOOKUP) |
| ⑧ | **선 그래프** | 시간에 따른 추이 |
| ⑨ | **상관계수** | -1 ~ +1, 같이 움직이는 정도 |
| ⑩ | **상관 ≠ 인과** | 평균만 봐도 안 되고, 이상치를 항상 지워도 안 됩니다 |

---
# 마지막 연습 — 빈칸 여덟 개

앞을 보지 말고 채워 보세요. 막히면 위로 올라가서 확인하면 됩니다.

### 문제 1. 엑셀 파일을 불러오는 함수는?

In [ ]:
# xlsx = pd.read_excel('파일이름.xlsx')
print('read_excel')

### 문제 2. 열별 결측치 개수를 세세요

In [ ]:
orders.isna().sum().head()

### 문제 3. 세부 품목(`Sub-Category`)별 이익 합계를 구하세요

In [ ]:
orders.groupby('Sub-Category')['Profit'].sum().round(0).sort_values()

### 문제 4. 위 결과에서 **적자인 품목만** 남기세요

In [ ]:
profit_by_item = orders.groupby('Sub-Category')['Profit'].sum()

profit_by_item[profit_by_item < 0]

### 문제 5. 주문 표와 반품 표를 `Order ID` 로 합치세요

In [ ]:
merged = pd.merge(orders, returns, on='Order ID', how='left')

print(merged.shape)

### 문제 6. 지역별 매출을 **막대그래프**로 그리세요

In [ ]:
plt.figure(figsize=(7, 3))
orders.groupby('Region')['Sales'].sum().plot(kind='bar')
plt.title('Sales by Region')
plt.show()

### 문제 7. 매출과 이익의 상관계수를 구하세요

In [ ]:
print(round(orders['Sales'].corr(orders['Profit']), 3))

### 문제 8. 코드가 아닙니다

**회사에 돌아가서 이번 주에 열어 볼 엑셀 파일 하나**를 정하세요.
아래 네 가지를 그 파일에 대고 채워 보세요.

| | 적어 보기 |
|---|---|
| 파일 이름 | |
| 한 행이 무엇인가 | |
| 묶어 보고 싶은 기준 (`groupby`) | |
| 그려 보고 싶은 그림 | |

코드가 기억나지 않으면 **각 교시 노트북**에 다 있습니다.
그리고 오늘 6교시에서 했듯이, **막히면 물어보면 됩니다.**